# Graphs I

## Graphs

A graph is a structure composed of nodes/vertexes ($V$) which are related between each other with
edges ($(u,v)\in E$):

```mermaid
---
title: G = (V,E)
---
stateDiagram-v2
    direction LR
    1 --> 2
    2 --> 4
    2 --> 3
    3 --> 6
    5 --> 4
    5 --> 3
    5 --> 6
```

We call the graph G undirected if $(u,v)\in E \Leftrightarrow (v,u)\in E$, i.e. the order of the vertexes defining an edge **doesn't matter**.

If the order is important, the graph is directed.

Additionally, vertexes or edges could have weights associated:
$$w_v = f(v), w_e = f((u,v))$$

### Working with Graphs

In general, we need to traverse the graph:
- Iterate over nodes.
- Given a node, iterate over outgoing/incoming edges.
- Given a edge, extract the associated nodes.

### Adjacency Matrix

A possible representation of a graph is using an *Adjacency Matrix*. A graph is represented with a matrix $M$ such as, if nodex $v_i$ and $v_j$ are connected, e.g. $(v_i,v_j)\in V$, then $M[i,j] \neq 0$, otherwise $M[i,j] = 0$.
- For an undirected graph, the matrix $M$ is symmetric.
- Edge weights can be stored in entries of the adjacency matrix instead of the value $1$.

For example, the graph
```mermaid
stateDiagram-v2
    direction LR
    1 --> 2
    2 --> 4
    2 --> 3
    3 --> 6
    5 --> 4
    5 --> 3
    5 --> 6
```
Can be represented by the matrix
```Python
G = [
    [0, 1, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 1],
    [0, 0, 0, 0, 0, 0],
]

### Adjacency List

A different representation of a graph is by adjacency lists.
- An array of lists $L[]$, with the number of elements equal the number of nodes in the graph.
- the $i$th entry of the lust corresponds to a list all nodes $j$ such that $(v_i,v_j)\in V$

$$L[i] = \text{list}(\dots,j,\dots) \Rightarrow (v_i,v_j)\in E$$

```mermaid
stateDiagram-v2
    direction LR
    0 --> 2
    2 --> 3
    3 --> [*]
```
```mermaid
stateDiagram-v2
    direction LR
    1 --> 2
    2 --> [*]
```
```mermaid
stateDiagram-v2
    direction LR
    2 --> 3
    3 --> [*]
```
```mermaid
stateDiagram-v2
    direction LR
    3 --> [*]
```

For example, the graph
```mermaid
stateDiagram-v2
    direction LR
    1 --> 2
    2 --> 4
    2 --> 3
    3 --> 6
    5 --> 4
    5 --> 3
    5 --> 6
```
Can be represented by the list
```Python
G = [
    [1],
    [2, 3],
    [5],
    [],
    [2, 5],
    [],
]
```
Or to make it more efficient:
```Python
G = {
    0: {1},
    1: {2, 3},
    3: {5},
    4: set(),
    5: {2, 5},
    6: set(),
}
```

### Adjacency Matrix vs Adjacency Lists

1. Matrix
    - Checking if two vertexes are connected takes $O(1)$ time.
    - Storing the graph requires $O(|V|^2)$ memory space (worst case).
2. Lists
    - Checking if two nodes are connected takes $O(|V|)$ time.
    - Storing the graph requires $O(|V|+|E|)$ space.

### Implicit Graph

In cases were the graph is too big, or even infinite, it is convenient to calculate the neighbors of a node "on-the-fly" rather than storing them directly.

Other case of implicit graph is when the nodes and edges can be calculated following some specific rules.

#### Implicit Graph Examples

- A 2D maze composed of $M\times N$ cells, in which we know if a cell $(i,j)$ is occupied or free.
![](images/maze.png)
- Numbers in an interval $[m,n]$ in which two numbers are related if they are multiples.
![](images/interval.png)

### Representing a General Graph in Python

```Python
class node:
    def __init__(self, key, data=None, color=None):
        self.key = key
        self.data = data
        self.color = color

class graph:
    def __init__(self, ...):
        """Initialize the necessary parameters"""
        pass
    def nodes(self):
        """Yield all possible nodes"""
        pass
    def neighbors(self, v):
        """Yield all neighbors starting at node v"""
        for ... in ...:
            do_something(...)
            yield node(k, d, c)
```

Or a `dict`:

```Python
node = dict(key="...", data=..., color=...)
````
Or just the `key`.

And the graph modeleted by a function `neighbors`:
```Python
def neighbors(v):
    """Yield all neighbors starting at node v"""
    for ... in ...:
        do_something(...)
        yield dict(...)
```
And the properties modeled using `dict`:
```Python
color = {1: "RED", 2: "BLUE", ...}
```

For example, the graph
```mermaid
stateDiagram-v2
    direction LR
    1 --> 2
    2 --> 4
    2 --> 3
    3 --> 6
    5 --> 4
    5 --> 3
    5 --> 6
```
```Python
# nodes = 0, 1, 2, 3, 4, 5
def neighbors(v):
    for n in G[v]:
        yield n

### Graph Exploring

Most graph problems are about finding a particular graph or a structure within the graph that has some particular property $P$.

There are 2 main search strategies:
- Deep First Search (DFS)
- Breath First Search (BFS)

And a third for special cases:
- Best First Search

#### Deep First Search (DFS)

Explores the graph, searching for property $P$, first following edges until no further edge is available.

##### DFS (Recursive, Backtracking)

In [1]:
import enum
class color(enum.Enum):
    WHITE = 0
    GREY = 1
    BLACK = 2

def DFS(G, v, P):
    v.color = color.GREY
    for u in G.neighbors(v):
        if u.color == color.WHITE:
            if P(G, u):
                return True # Or "return u" if we are searching for a node
        res = DFS(G, u, P)
        if res:
            return True # Or "return res" if we are searching for a node
    v.color = color.BLACK
    return False # Or "return None" if we are searching for a node

##### DFS (Iterative)

In [2]:
from collections import deque

def DFS(G, s, P):
    next_vertexes = deque([s])
    while len(next_vertexes) > 0:
        v = next_vertexes.pop()
        v.color = color.GREY
        if P(G, v):
            return True # or "return v" if we are searching for a node
        else:
            for u in G.neighbors(v):
                if u.color == color.WHITE:
                    next_vertexes.append(u)
        v.color = color.BLACK
    return False # or "return None" if we are searching for a node

##### Example

![](images/dfs.png)

#### Breath First Search (BFS)

Evaluates all sibling nodes and then expand to their child nodes.

##### BFS

In [3]:
def BFS(G, s, P):
    next_vertexes = deque([s])
    while len(next_vertexes) > 0:
        v = next_vertexes.popleft()
        v.color = color.GREY
        if P(G, v):
            return True # or "return v" if we are searching for a node
        else:
            for u in G.neighbors(v):
                if u.color == color.WHITE:
                    u.color = color.GREY
                    next_vertexes.append(u)
        v.color = color.BLACK
    return False # or "return None" if we are searching for a node

##### Example

![](images/bfs.png)

### Related Problems

- Find connected components.
- Flood fill.
- Topological sort.
- Articulation points.
- Strongly connected components.
- Check for a birpatite graph.

#### Connected Components

Run `dfs`or `bfs` from a starting node, if after finishing its execution, white nodes remain, a new component has been detected. Repeat until no white nodes remain.

#### Flood fill, or count the size of independent components

Modify the recursive `dfs`to return the size of each component.

#### Topological sort

Given an acyclic directed graph, find some order of the nodes $u_i,u_2,\dots,u_n$ such that if $u_i$ appears after node $u_j$ in the order, there is a directed path between $u_i \rightarrow u_j$ or $u_i$ and $u_j$ are independent components of the graph.
![](images/topological-sort.png)

Save closed nodes in a "global" list. The topological order is given by reversing this list.

In [4]:
def topological_sort_dfs(G, L, v):
    v.color = color.GREY
    for u in G.neighbor(v):
        if u.color == color.WHITE:
            if not topological_sort_dfs(G, L, u):
                return False
            elif u.color == color.BLACK:
                return False
    v.color = color.BLACK
    L.prepend(v)
    return True

#### Articulation Points

Simplest algorithm, use `dfs` or `bfs` to count for connected components (CC). Then for each vertex $v$, remove the vertex and count the number of CC's. If it increases, then $v$ is an articulation point.

Using `dfs`:
- For every node `v`, track `num(v)`as the iteration number when node v is first visited.
- For every node `v`, track `low(v)`as the lowest `num(v)` reachable from the exploration starting at `v`, not taking into account the parent of `v`.
- If at the end of `dfs`, when we are at node `u` with neighbor `v`, if `low(v) >= num(u)` then `u` is an articulation point.
- Similarly, the edge `uv` is a bridge if `low(v) > num(u)`

#### Strongly Connected Components

![](images/scc.png)
Number the nodes, and use `dfs` to traverse the graph keeping track of the lowest number seen by a node.

In [5]:
node_count = 0
def stronly_cc_dfs(G, v):
    global node_count
    v.color = color.GREY
    v.min_seen = v.key = node_count
    node_count += 1
    for u in G.neighbors(v):
        if u.color == color.WHITE:
            stronly_cc_dfs(G, u)
        v.min_seen = min(v.min_seen, u.min_seen)
    v.color = color.BLACK

#### Bipartite (or 2 Colorable) Graph Check

A bipartite graph can be divided into two components with edges only between the components and not within the components.

![](images/bipartite.png)

Use `bfs` with two new colors, coloring neighbor nodes with different colors. If two adjacent nodes share the color, the graph is not 2-clorable/bipartite.

In [6]:
def bipartite_BFS(G, s, P):
    s.color = color.RED
    next_vertexes = [s]
    while len(next_vertexes) > 0:
        v = next_vertexes.pop()
        for u in G.neighbors(v):
            if u.color == color.WHITE:
                u.color = (color.BLUE if v.color == color.RED else color.RED)
                next_vertexes.prepend(u)
            elif u.color == v.color:
                return False
    return True
